# Notebook 4 — Export Tuned Model for Streamlit Deployment
## BRFSS 2022 Heart Disease Prediction

**Output files:**
- `model_xgb_only.pkl` — XGBoost model (no sklearn pipeline)
- `feature_names.pkl` — list of 50 features
- `model_metadata.json` — threshold + metrics + scaler + Optuna best params


### Changelog
- **[FIX]** `scale_pos_weight=1` (was SPW≈10.24): SMOTE balances to 1:1 already;
  using SPW simultaneously is a double correction (see NB3B ablation Section 3).
- **[FIX]** Now re-runs Optuna (25 trials, same seed as NB3B) and trains with
  Optuna best params. Previously metadata claimed 'Optuna 25 trials' but model
  was actually trained with hardcoded baseline params — now consistent.
- **[FIX]** Metadata updated: `optuna_best_params` added for reproducibility.

In [ ]:
!pip install -q imbalanced-learn xgboost scikit-learn optuna

import warnings, os, json
warnings.filterwarnings('ignore')
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, precision_score, f1_score, confusion_matrix)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

SEED = 42
OUT  = '/content/nb4_outputs'
os.makedirs(OUT, exist_ok=True)
print('✓ Setup done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.2 MB/s eta 0:00:00
✓ Setup done


## 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

dm = pd.read_csv('brfss2022_clean.csv')
X  = dm.drop(columns=['target'])
y  = dm['target'].astype(int)
FEAT_NAMES = list(X.columns)

CONT_COLS = ['Sleep_hours', 'PhysHealth_days', 'MentHealth_days']
BIN_COLS  = [c for c in X.columns if c not in CONT_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

SPW = (y_train==0).sum() / (y_train==1).sum()

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), CONT_COLS),
], remainder='passthrough')

print(f'✓ Dataset  : {len(dm):,} records | CVD rate: {y.mean()*100:.2f}%')
print(f'  Train/Test: {len(X_train):,} / {len(X_test):,}')
print(f'  SPW={SPW:.2f}  [reference only — NOT used, SMOTE handles balancing]')

Saving brfss2022_clean.csv to brfss2022_clean.csv
✓ Dataset  : 340,154 records | CVD rate: 8.89%
  Train/Test: 272,123 / 68,031
  SPW=10.24  [reference only — NOT used, SMOTE handles balancing]


## 3. Optuna Tuning (25 trials — identical to NB3B)

Same search space + seed → same best params as NB3B. Re-running here ensures
the exported model is truly the Optuna-tuned version.

In [ ]:
def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 100, 600),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
        gamma            = trial.suggest_float('gamma', 0.0, 5.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        scale_pos_weight = 1,
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', random_state=SEED, n_jobs=-1,
    )
    pipe = ImbPipeline([
        ('pre',   clone(preprocessor)),
        ('smote', SMOTE(random_state=SEED)),
        ('clf',   XGBClassifier(**params))
    ])
    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(pipe, X_train, y_train,
                           cv=cv5, scoring='roc_auc', n_jobs=1).mean()

print('Running Optuna 25 trials (same as NB3B)...')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=25, show_progress_bar=True)

print(f'\n✓ Best CV AUC : {study.best_value:.4f}')
print('  Best params :')
for k, v in study.best_params.items():
    print(f'    {k:25s}: {v}')

Running Optuna 25 trials (same as NB3B)...


  0%|          | 0/25 [00:00<?, ?it/s]


✓ Best CV AUC : 0.8282
  Best params :
    n_estimators             : 532
    max_depth                : 3
    learning_rate            : 0.20620175200624824
    subsample                : 0.8354841558571261
    colsample_bytree         : 0.7412737715466858
    min_child_weight         : 6
    gamma                    : 0.6975488466566845
    reg_alpha                : 0.008696836819616664
    reg_lambda               : 0.0004535982683292706


## 4. Train Final Pipeline with Best Params

In [ ]:
best_p = study.best_params.copy()
best_p.update({'scale_pos_weight': 1, 'objective': 'binary:logistic',
               'eval_metric': 'logloss', 'tree_method': 'hist',
               'random_state': SEED, 'n_jobs': -1})

pipeline = ImbPipeline([
    ('pre',   clone(preprocessor)),
    ('smote', SMOTE(random_state=SEED)),
    ('clf',   XGBClassifier(**best_p))
])

print('Training final pipeline (Optuna best params, SMOTE + SPW=1)...')
pipeline.fit(X_train, y_train)
ypr = pipeline.predict_proba(X_test)[:, 1]
print(f'✓ Trained | AUC: {roc_auc_score(y_test, ypr):.4f}')

Training final pipeline (Optuna best params, SMOTE + SPW=1)...
✓ Trained | AUC: 0.8275


## 5. G-mean Threshold Scan

In [ ]:
rows = []
for t in np.arange(0.01, 1.0, 0.01):
    yt_ = (ypr >= t).astype(int)
    if yt_.sum() == 0: continue
    tn, fp, fn, tp_ = confusion_matrix(y_test, yt_).ravel()
    rec  = tp_/(tp_+fn);  spec = tn/(tn+fp)
    prec = precision_score(y_test, yt_, zero_division=0)
    gm   = np.sqrt(rec*spec)
    fb2  = (1+4)*prec*rec/(4*prec+rec+1e-9)
    rows.append(dict(t=round(t,2), recall=rec, spec=spec, prec=prec, gm=gm, fb2=fb2))

td  = pd.DataFrame(rows)
bgm = td.loc[td['gm'].idxmax()]
FINAL_T = float(bgm['t'])
AUC     = roc_auc_score(y_test, ypr)

print(f'✓ G-mean optimal threshold : {FINAL_T}')
print(f'  Sensitivity : {bgm.recall:.4f}')
print(f'  Specificity : {bgm.spec:.4f}')
print(f'  Precision   : {bgm.prec:.4f}')
print(f'  G-mean      : {bgm.gm:.4f}')
print(f'  AUC         : {AUC:.4f}')

✓ G-mean optimal threshold : 0.12
  Sensitivity : 0.7883
  Specificity : 0.7198
  Precision   : 0.2155
  G-mean      : 0.7533
  AUC         : 0.8275


## 6. Export Files

In [ ]:
clf    = pipeline.named_steps['clf']
pre    = pipeline.named_steps['pre']
scaler = pre.named_transformers_['scale']

joblib.dump(clf, f'{OUT}/model_xgb_only.pkl')
print(f'✓ model_xgb_only.pkl ({os.path.getsize(f"{OUT}/model_xgb_only.pkl")/1024/1024:.1f} MB)')

joblib.dump(FEAT_NAMES, f'{OUT}/feature_names.pkl')
print(f'✓ feature_names.pkl  ({len(FEAT_NAMES)} features)')

yp_opt = (ypr >= FINAL_T).astype(int)
tn, fp, fn, tp_ = confusion_matrix(y_test, yp_opt).ravel()

metadata = {
    'threshold':     FINAL_T,
    'feature_names': FEAT_NAMES,
    'cont_cols':     CONT_COLS,
    'bin_cols':      BIN_COLS,
    'metrics': {
        'sensitivity': round(float(tp_/(tp_+fn)), 4),
        'specificity': round(float(tn/(tn+fp)),   4),
        'gmean':       round(float(bgm['gm']),    4),
        'auc':         round(float(AUC),          4),
        'precision':   round(float(precision_score(y_test, yp_opt, zero_division=0)), 4),
        'f1':          round(float(f1_score(y_test, yp_opt, zero_division=0)),        4),
    },
    'training_info': {
        'dataset':            'BRFSS 2022',
        'n_samples':          int(len(dm)),
        'n_features':         int(len(FEAT_NAMES)),
        'cvd_rate':           round(float(y.mean()), 4),
        'model':              'XGBoost + SMOTE, scale_pos_weight=1 (Config B)',
        'tuning':             f'Optuna {len(study.trials)} trials (TPESampler, seed={SEED})',
        'optuna_best_auc':    round(study.best_value, 4),
        'optuna_best_params': study.best_params,
    },
    'scaler': {
        'cont_cols': CONT_COLS,
        'mean':      [float(v) for v in scaler.mean_],
        'std':       [float(v) for v in scaler.scale_],
    }
}

with open(f'{OUT}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

with open(f'{OUT}/model_metadata.json') as f:
    chk = json.load(f)
print(f'✓ model_metadata.json')
print(f'  threshold   : {chk["threshold"]}')
print(f'  sensitivity : {chk["metrics"]["sensitivity"]}')
print(f'  specificity : {chk["metrics"]["specificity"]}')
print(f'  precision   : {chk["metrics"]["precision"]}')
print(f'  gmean       : {chk["metrics"]["gmean"]}')
print(f'  auc         : {chk["metrics"]["auc"]}')
print(f'  model       : {chk["training_info"]["model"]}')
print(f'  tuning      : {chk["training_info"]["tuning"]}')
print()
print('✅ All 3 files ready!')

✓ model_xgb_only.pkl (0.6 MB)
✓ feature_names.pkl  (50 features)
✓ model_metadata.json
  threshold   : 0.12
  sensitivity : 0.7883
  specificity : 0.7198
  precision   : 0.2155
  gmean       : 0.7533
  auc         : 0.8275
  model       : XGBoost + SMOTE, scale_pos_weight=1 (Config B)
  tuning      : Optuna 25 trials (TPESampler, seed=42)

✅ All 3 files ready!


## 7. Sanity Check — Same logic as app.py

In [ ]:
clf_chk  = joblib.load(f'{OUT}/model_xgb_only.pkl')
feat_chk = joblib.load(f'{OUT}/feature_names.pkl')
with open(f'{OUT}/model_metadata.json') as f:
    meta_chk = json.load(f)

sc = meta_chk['scaler']
FINAL_ORDER = sc['cont_cols'] + [c for c in feat_chk if c not in sc['cont_cols']]

sample = {name: 0 for name in feat_chk}
sample.update({'Age_65_plus': 1, 'Male': 1, 'Stroke': 1, 'COPD': 1,
               'Diabetes_yes': 1, 'GenHealth_poor': 1,
               'Sleep_hours': 5.0, 'PhysHealth_days': 20.0, 'MentHealth_days': 10.0})

X_s = pd.DataFrame([sample])[FINAL_ORDER]
for i, col in enumerate(sc['cont_cols']):
    X_s[col] = (X_s[col] - sc['mean'][i]) / (sc['std'][i] + 1e-8)

prob = float(clf_chk.predict_proba(X_s.values)[0, 1])
pred = 'HIGH RISK ⚠️' if prob >= meta_chk['threshold'] else 'LOW RISK ✅'
print(f'Profile: Age65+, Male, Stroke, COPD, Diabetes, Poor health')
print(f'Probability : {prob:.4f}  |  Threshold: {meta_chk["threshold"]}  |  {pred}')
print('✓ Model loads and predicts correctly')

Profile: Age65+, Male, Stroke, COPD, Diabetes, Poor health
Probability : 0.6569  |  Threshold: 0.12  |  HIGH RISK ⚠️
✓ Model loads and predicts correctly


## 8. Download

In [ ]:
from google.colab import files
for fname in ['model_xgb_only.pkl', 'feature_names.pkl', 'model_metadata.json']:
    files.download(f'{OUT}/{fname}')
    print(f'⬇️  {fname}')
print('\n✅ Copy 3 files vào cùng folder app.py → streamlit run app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  model_xgb_only.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  feature_names.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  model_metadata.json

✅ Copy 3 files vào cùng folder app.py → streamlit run app.py
